# Comparing Tree-Based Ensemble Methods for Classification

This project builds and compares eight tree-based classification 
approaches on the same dataset and the same leak-safe pipeline: a 
single decision tree, two parallel-ensemble methods (Bagging, Random 
Forest), two sequential-ensemble methods (AdaBoost, Gradient Boosting), 
each tested both untuned and tuned, plus XGBoost as an industry-standard 
alternative implementation of the same boosting idea. The goal isn't 
just finding the highest score - it's understanding what each method's 
underlying mechanism does differently, and choosing a final model based 
on the metric that actually matters for the problem, not just whichever 
number is highest.

## Setup

In [1]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import StratifiedKFold, cross_validate, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

df = pd.read_csv('https://raw.githubusercontent.com/pandas-dev/pandas/master/doc/data/titanic.csv')
numeric_features = ['Age', 'Fare', 'SibSp', 'Parch']
categorical_features = ['Sex', 'Embarked']

numeric_transformer = Pipeline([('imputer', SimpleImputer(strategy='median'))])
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first', handle_unknown='ignore'))
])
preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features),
    ('pass', 'passthrough', ['Pclass'])
])

features = numeric_features + categorical_features + ['Pclass']
X = df[features]
y = df['Survived']
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

## Tuning Gradient Boosting and XGBoost

Both boosting implementations are tuned jointly across n_estimators, 
max_depth, and learning_rate using GridSearchCV, rather than sweeping 
each parameter separately - these parameters interact (a shallower tree 
can pair with a larger learning_rate to reach a similar result as a 
deeper tree with a smaller one), so searching them together finds 
combinations a one-at-a-time sweep would miss.

In [2]:
param_grid = {
    'model__n_estimators': [50, 100, 200],
    'model__max_depth': [2, 3, 5],
    'model__learning_rate': [0.01, 0.1, 0.5]
}

gb_pipeline = Pipeline([('preprocessor', preprocessor), ('model', GradientBoostingClassifier(random_state=42))])
grid_search_gb = GridSearchCV(gb_pipeline, param_grid=param_grid, scoring='f1', cv=skf, n_jobs=-1)
grid_search_gb.fit(X, y)
print("Gradient Boosting best params:", grid_search_gb.best_params_)

xgb_pipeline = Pipeline([('preprocessor', preprocessor), ('model', XGBClassifier(random_state=42, eval_metric='logloss'))])
grid_search_xgb = GridSearchCV(xgb_pipeline, param_grid=param_grid, scoring='f1', cv=skf, n_jobs=-1)
grid_search_xgb.fit(X, y)
print("XGBoost best params:", grid_search_xgb.best_params_)

Gradient Boosting best params: {'model__learning_rate': 0.5, 'model__max_depth': 2, 'model__n_estimators': 100}
XGBoost best params: {'model__learning_rate': 0.5, 'model__max_depth': 3, 'model__n_estimators': 50}


## Building and Evaluating All Methods

In [4]:
models = {
    'Single Tree (depth=3)': DecisionTreeClassifier(max_depth=3, random_state=42),
    'Bagging (100 trees)': BaggingClassifier(estimator=DecisionTreeClassifier(random_state=42), n_estimators=100, random_state=42),
    'Random Forest (100 trees)': RandomForestClassifier(n_estimators=100, random_state=42),
    'AdaBoost (stumps, default)': AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1, random_state=42), n_estimators=100, random_state=42),
    'Gradient Boosting (default)': GradientBoostingClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting (tuned)': GradientBoostingClassifier(**{k.replace('model__', ''): v for k, v in grid_search_gb.best_params_.items()}, random_state=42),
    'XGBoost (default)': XGBClassifier(n_estimators=100, max_depth=3, learning_rate=0.1, random_state=42, eval_metric='logloss'),
    'XGBoost (tuned)': XGBClassifier(**{k.replace('model__', ''): v for k, v in grid_search_xgb.best_params_.items()}, random_state=42, eval_metric='logloss'),
}

results = []
for name, model in models.items():
    pipeline = Pipeline([('preprocessor', preprocessor), ('model', model)])
    cv = cross_validate(pipeline, X, y, cv=skf, scoring=['accuracy', 'recall', 'precision', 'f1'])
    results.append({
        'Method': name,
        'Accuracy': cv['test_accuracy'].mean(),
        'Recall': cv['test_recall'].mean(),
        'Precision': cv['test_precision'].mean(),
        'F1': cv['test_f1'].mean()
    })

results_df = pd.DataFrame(results).round(4)
print(results_df)

                        Method  Accuracy  Recall  Precision      F1
0        Single Tree (depth=3)    0.8249  0.7102     0.8102  0.7558
1          Bagging (100 trees)    0.8204  0.7572     0.7734  0.7638
2    Random Forest (100 trees)    0.8182  0.7485     0.7745  0.7602
3   AdaBoost (stumps, default)    0.8115  0.7394     0.7660  0.7494
4  Gradient Boosting (default)    0.8328  0.7105     0.8303  0.7650
5    Gradient Boosting (tuned)    0.8384  0.7454     0.8177  0.7796
6            XGBoost (default)    0.8305  0.7047     0.8294  0.7611
7              XGBoost (tuned)    0.8384  0.7514     0.8129  0.7807


## Results

| Method | Accuracy | Recall | Precision | F1 |
|---|---|---|---|---|
| Single Tree (depth=3) | 0.8249 | 0.7102 | 0.8102 | 0.7558 |
| Bagging (100 trees) | 0.8204 | **0.7572** | 0.7734 | 0.7638 |
| Random Forest (100 trees) | 0.8182 | 0.7485 | 0.7745 | 0.7602 |
| AdaBoost (stumps, default) | 0.8115 | 0.7394 | 0.7660 | 0.7494 |
| Gradient Boosting (default) | 0.8328 | 0.7105 | 0.8303 | 0.7650 |
| Gradient Boosting (tuned) | **0.8384** | 0.7454 | 0.8177 | 0.7796 |
| XGBoost (default) | 0.8305 | 0.7047 | 0.8294 | 0.7611 |
| **XGBoost (tuned)** | **0.8384** | **0.7514** | 0.8129 | **0.7807** |

Tuned XGBoost leads on accuracy (tied with tuned Gradient Boosting), 
recall, and F1. No untuned method beats either tuned boosting model on 
F1, confirming that joint hyperparameter search - tuning n_estimators, 
max_depth, and learning_rate together rather than one at a time - finds 
real improvements a simpler sweep would miss.

The precision/recall split across methods reflects each one's underlying 
mechanism. The single tree and untuned Gradient Boosting/XGBoost both 
lean cautious: high precision, lower recall - confident when they 
predict "Survived," but missing more real survivors along the way. 
Bagging and Random Forest lean the opposite way: averaging across many 
trees makes them more willing to call borderline cases "Survived," 
lifting recall at some cost to precision. AdaBoost, using shallow stumps 
at its default learning_rate, underperforms across the board - a likely 
sign its default settings aren't well-suited to this dataset, though it 
was not separately tuned here. Tuned Gradient Boosting and tuned XGBoost 
both improve recall meaningfully over their untuned versions without 
giving up as much precision as Bagging or Random Forest did to get 
there, which is why they lead on F1.

**No single method is universally "best" - the right choice depends on 
which metric actually matters for the problem being solved.** If missing 
a real positive case is the more costly error, recall should drive the 
decision. If a false alarm is more costly, precision matters more. If 
both types of mistakes carry meaningful cost and neither should be 
favored outright, F1 is the metric to optimize for, since it's built 
specifically to reward that balance rather than either extreme.

For this dataset, both tuned boosting methods - Gradient Boosting and 
XGBoost - come out ahead on that balance, landing close together at the 
top of the table (F1 of 0.7796 and 0.7807) with no other method coming 
close on F1 without tuning. Optimizing for accuracy alone, without first 
deciding which error costs more or whether both matter, risks picking a 
model that looks good on paper but is quietly wrong for the actual use 
case.

Two other factors matter beyond the numbers in this table. The single 
decision tree remains the easiest model here to explain to a 
non-technical stakeholder - its decision path can be read directly off 
a diagram, split by split, with no need to interpret coefficients or 
aggregate importances across many trees. XGBoost, on the other hand, has 
real advantages that matter more in production than in a comparison 
like this one: it trains faster on large datasets thanks to optimized, 
parallelized tree-building, has regularization built directly into its 
objective function to help control overfitting, and can handle missing 
values natively without requiring a separate imputation step - a genuine 
engineering convenience on messier, real-world data.